In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
train_path = PROCESSED_DATA_DIR / "train.json"
validation_path = PROCESSED_DATA_DIR / "validation.json"
test_path = PROCESSED_DATA_DIR / "test.json"

In [3]:
import json

# Load the finalized splits
with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(validation_path, "r", encoding="utf-8") as f:
    val_data = json.load(f)

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

Train: 34185
Validation: 4268
Test: 4208


In [5]:
train_data[0]

{'id': 'FinQA-30K_dp11842.pdf_6',
 'question': "What is the primary difference in impact between parental income and parental wealth regarding a child's educational outcomes?",
 'answer': 'Parental income has a larger effect on whether a child attends college, whereas parental wealth has a significant effect on whether the child graduates from college.',
 'context': '',
 'table': '',
 'domain': 'Personal Finance and Wealth Management',
 'question_type': 'factual',
 'reasoning': '',
 'source_dataset': 'FinQA-30K',
 'source_id': 'dp11842.pdf_6'}

In [7]:
# Show one example from each source dataset
shown = set()

for record in train_data:
    source = record["source_dataset"]

    if source not in shown:
        print("\n" + "=" * 80)
        print("SOURCE:", source)
        print("=" * 80)

        print("Question:", record["question"])
        print("Answer:", record["answer"])
        print("Context:", bool(record["context"]))
        print("Table:", bool(record["table"]))
        print("Reasoning:", bool(record["reasoning"]))
        print('question_type:' , record["question_type"])

        shown.add(source)

    if len(shown) == 3:
        break


SOURCE: FinQA-30K
Question: What is the primary difference in impact between parental income and parental wealth regarding a child's educational outcomes?
Answer: Parental income has a larger effect on whether a child attends college, whereas parental wealth has a significant effect on whether the child graduates from college.
Context: False
Table: False
Reasoning: False
question_type: factual

SOURCE: FinQA
Question: for the fourth quarter of 2016 , what was the total amount spent to repurchase shares ( in thousands ) ?\\n
Answer: 503807
Context: True
Table: True
Reasoning: True
question_type: 

SOURCE: ConvFinQA
Question: between 2000 and 2001 , what was the percent increase of unrealized gains?
Answer: 405.3%
Context: True
Table: True
Reasoning: True
question_type: conversational


In [8]:
from collections import defaultdict

# Keys we care about
keys = ["question", "answer", "context", "table", "reasoning", "question_type"]

# Store counts: source -> key -> count
counts = defaultdict(lambda: defaultdict(int))

for record in train_data:
    source = record["source_dataset"]
    
    # Always count presence of question & answer (they should always exist)
    counts[source]["question"] += 1
    counts[source]["answer"] += 1
    
    # For the boolean-like fields – count how many are True / non-empty
    if record.get("context"):
        counts[source]["context"] += 1
    if record.get("table"):
        counts[source]["table"] += 1
    if record.get("reasoning"):
        counts[source]["reasoning"] += 1
    
    # question_type – count only when it is non-empty
    if record.get("question_type"):
        counts[source]["question_type"] += 1

# Pretty print
for source in ["FinQA-30K", "FinQA", "ConvFinQA"]:
    print("\n" + "=" * 60)
    print(f"SOURCE: {source}")
    print("=" * 60)
    for k in keys:
        print(f"{k}: {counts[source][k]}")


SOURCE: FinQA-30K
question: 28078
answer: 28078
context: 0
table: 0
reasoning: 0
question_type: 28078

SOURCE: FinQA
question: 4589
answer: 4589
context: 4589
table: 4589
reasoning: 4589
question_type: 0

SOURCE: ConvFinQA
question: 1518
answer: 1518
context: 1518
table: 1518
reasoning: 1518
question_type: 1518


In [9]:
def format_sft_example(record):
    parts = []

    # Context
    if record["context"]:
        parts.append(f"### Context\n{record['context']}")

    # Table
    if record["table"]:
        parts.append(f"### Table\n{record['table']}")

    # Question
    parts.append(f"### Question\n{record['question']}")

    # Combine input sections
    prompt = "\n\n".join(parts)

    # Answer
    return {
        "prompt": prompt,
        "answer": record["answer"]
    }

In [10]:
for source in ["FinQA-30K", "FinQA", "ConvFinQA"]:

    record = next(
        r for r in train_data
        if r["source_dataset"] == source
    )

    example = format_sft_example(record)

    print("\n" + "=" * 80)
    print(source)
    print("=" * 80)

    print(example["prompt"])
    print("\n### Answer")
    print(example["answer"])


FinQA-30K
### Question
What is the primary difference in impact between parental income and parental wealth regarding a child's educational outcomes?

### Answer
Parental income has a larger effect on whether a child attends college, whereas parental wealth has a significant effect on whether the child graduates from college.

FinQA
### Context
part ii item 5 2013 market for registrant 2019s common equity , related stockholder matters and issuer purchases of equity securities ( a ) ( 1 ) our common stock is listed on the new york stock exchange and is traded under the symbol 201cpnc . 201d at the close of business on february 16 , 2017 , there were 60763 common shareholders of record .
holders of pnc common stock are entitled to receive dividends when declared by the board of directors out of funds legally available for this purpose .
our board of directors may not pay or set apart dividends on the common stock until dividends for all past dividend periods on any series of outstanding

In [13]:
train_sft = [
    format_sft_example(record)
    for record in train_data
]

validation_sft = [
    format_sft_example(record)
    for record in val_data
]

test_sft = [
    format_sft_example(record)
    for record in test_data
]

print("Train      :", len(train_sft))
print("Validation :", len(validation_sft))
print("Test       :", len(test_sft))

Train      : 34185
Validation : 4268
Test       : 4208


In [26]:
train_sft[0]

{'prompt': "### Question\nWhat is the primary difference in impact between parental income and parental wealth regarding a child's educational outcomes?",
 'answer': 'Parental income has a larger effect on whether a child attends college, whereas parental wealth has a significant effect on whether the child graduates from college.'}

In [27]:
# saving the SFT Datasets

train_sft_path = PROCESSED_DATA_DIR / "train_sft.json"
validation_sft_path = PROCESSED_DATA_DIR / "validation_sft.json"
test_sft_path = PROCESSED_DATA_DIR / "test_sft.json"

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2
        )


save_json(train_sft, train_sft_path)
save_json(validation_sft, validation_sft_path)
save_json(test_sft, test_sft_path)

print("SFT datasets saved:")
print(train_sft_path)
print(validation_sft_path)
print(test_sft_path)


SFT datasets saved:
d:\ai_project\FinGuide-AI\data\processed\train_sft.json
d:\ai_project\FinGuide-AI\data\processed\validation_sft.json
d:\ai_project\FinGuide-AI\data\processed\test_sft.json


In [28]:
for path in [
    train_sft_path,
    validation_sft_path,
    test_sft_path
]:
    with open(path, "r", encoding="utf-8") as f:
        loaded = json.load(f)

    print(f"{path.name}: {len(loaded):,} records")

train_sft.json: 34,185 records
validation_sft.json: 4,268 records
test_sft.json: 4,208 records
